In [12]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import lightgbm as lgb
import joblib
from joblib import Parallel, delayed
import multiprocessing

import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
import gc

In [2]:
N_JOBS = multiprocessing.cpu_count()   # use ALL cpu cores for the profile loop
print(f"CPU cores available: {N_JOBS}")

CPU cores available: 16


In [13]:
recipes = pd.read_csv("resource/recipes.csv", usecols=["RecipeId", "Name", "RecipeIngredientParts", "Keywords", "AggregatedRating", "ReviewCount"])
reviews = pd.read_csv("resource/reviews.csv", usecols=["RecipeId", "AuthorId", "Rating"])

In [14]:
recipes = recipes.dropna(subset=["RecipeIngredientParts"]).copy()
reviews["AuthorId"] = reviews["AuthorId"].astype(str)

In [15]:
recipes["text"] = (
    recipes["RecipeIngredientParts"].fillna("") + " " +
    recipes["Keywords"].fillna("") + " " +
    recipes["Name"].fillna("")
).str.lower()

In [16]:
valid_recipes = set(recipes["RecipeId"])
reviews = reviews[reviews["RecipeId"].isin(valid_recipes)].copy()

In [17]:
recipe_ids = recipes["RecipeId"].unique()
recipe_id_to_idx = {rid: i for i, rid in enumerate(recipe_ids)}

In [18]:
user_ids = reviews["AuthorId"].unique()
user_id_to_idx = {uid: i for i, uid in enumerate(user_ids)}

In [19]:
recipes["recipe_idx"] = recipes["RecipeId"].map(recipe_id_to_idx)
reviews["recipe_idx"] = reviews["RecipeId"].map(recipe_id_to_idx)
reviews["user_idx"] = reviews["AuthorId"].map(user_id_to_idx)

In [20]:
recipes = recipes.sort_values("recipe_idx")

In [21]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    dtype=np.float32 # Save RAM immediately
)

In [22]:
recipe_tfidf_matrix = vectorizer.fit_transform(recipes["text"])

In [12]:
rows = reviews["user_idx"].values
cols = reviews["recipe_idx"].values
data = reviews["Rating"].fillna(3).values.astype(np.float32)

In [13]:
user_recipe_matrix = sp.coo_matrix(
    (data, (rows, cols)),
    shape=(len(user_ids), len(recipe_ids))
).tocsr()

In [14]:
user_recipe_matrix = normalize(user_recipe_matrix, norm='l1', axis=1)

In [15]:
user_profiles_matrix = user_recipe_matrix.dot(recipe_tfidf_matrix)

In [16]:
del user_recipe_matrix
gc.collect()

0

In [17]:
agg_rating_arr = recipes["AggregatedRating"].fillna(0).values
rev_count_arr = np.log1p(recipes["ReviewCount"].fillna(0).values)

In [18]:
review_recipe_idxs = reviews["recipe_idx"].values
review_user_idxs = reviews["user_idx"].values

In [19]:
review_agg_rating = agg_rating_arr[review_recipe_idxs].reshape(-1, 1)
review_rev_count = rev_count_arr[review_recipe_idxs].reshape(-1, 1)
numeric_features = sp.csr_matrix(np.hstack([review_agg_rating, review_rev_count]), dtype=np.float32)

In [ ]:
del recipes
gc.collect()

In [20]:
chunk_size = 50000
total_rows = len(reviews)
x_chunks = []

In [21]:
for i in range(0, total_rows, chunk_size):
    # Slice mappings
    r_idx_chunk = review_recipe_idxs[i : i + chunk_size]
    u_idx_chunk = review_user_idxs[i : i + chunk_size]

    # Grab just the vectors for this chunk
    r_chunk = recipe_tfidf_matrix[r_idx_chunk]
    u_chunk = user_profiles_matrix[u_idx_chunk]

    # Multiply interaction just for this chunk
    i_chunk = r_chunk.multiply(u_chunk)
    n_chunk = numeric_features[i : i + chunk_size]

    # Horizontally stack
    combined_chunk = sp.hstack([r_chunk, u_chunk, i_chunk, n_chunk], format='csr', dtype=np.float32)
    x_chunks.append(combined_chunk)

In [22]:
print("Vstacking chunks...")
X = sp.vstack(x_chunks, format='csr')

Vstacking chunks...


In [23]:
del x_chunks, recipe_tfidf_matrix, user_profiles_matrix, numeric_features, review_recipe_idxs, review_user_idxs
gc.collect()

0

In [24]:
print(f"  Final Feature matrix shape: {X.shape}")

  Final Feature matrix shape: (1401963, 9002)


In [25]:
y = (reviews["Rating"] == 5).astype(np.int8).values
print(f"  5-star rate: {y.mean():.1%}")

  5-star rate: 72.2%


In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

In [31]:
import joblib
import scipy.sparse as sp
import numpy as np
from tqdm import tqdm
import os

def fast_save_sparse(matrix, prefix, pbar):
    """เซฟ Sparse Matrix แบบไม่บีบอัด (เร็วขึ้นมาก)"""
    # สร้างโฟลเดอร์เก็บแยกเพื่อความระเบียบ (เพราะ 1 matrix จะได้ 4 ไฟล์)
    if not os.path.exists(prefix): os.makedirs(prefix)

    pbar.set_postfix_str(f"Writing {prefix} data...")
    np.save(f"{prefix}/data.npy", matrix.data)
    np.save(f"{prefix}/indices.npy", matrix.indices)
    np.save(f"{prefix}/indptr.npy", matrix.indptr)
    np.save(f"{prefix}/shape.npy", np.array(matrix.shape))

print("Starting High-Speed Saving (Uncompressed)...")

total_files = 7
with tqdm(total=total_files, desc="Fast Saving", unit="file") as pbar:
    # 1. เซฟ Matrices แบบ Raw (เร็วขึ้น 3-5 เท่า)
    fast_save_sparse(X_train, "X_train_raw", pbar)
    pbar.update(1)

    fast_save_sparse(X_test, "X_test_raw", pbar)
    pbar.update(1)

    # 2. เซฟ Target (ปกติเร็วอยู่แล้ว)
    pbar.set_postfix_str("Saving y_train...")
    np.save("y_train.npy", y_train)
    pbar.update(1)

    pbar.set_postfix_str("Saving y_test...")
    np.save("y_test.npy", y_test)
    pbar.update(1)

    # 3. เซฟ Artifacts (ใช้ pickle protocol ล่าสุดเพื่อความเร็ว)
    pbar.set_postfix_str("Saving vectorizer...")
    joblib.dump(vectorizer, "tfidf_vectorizer.pkl", protocol=5)
    pbar.update(1)

    joblib.dump(user_id_to_idx, "user_id_to_idx.pkl", protocol=5)
    pbar.update(1)

    joblib.dump(recipe_id_to_idx, "recipe_id_to_idx.pkl", protocol=5)
    pbar.update(1)

print("Done! Speed was prioritized over disk space.")

Starting High-Speed Saving (Uncompressed)...


Fast Saving: 100%|██████████| 7/7 [01:39<00:00, 14.26s/file, Saving vectorizer...]       

Done! Speed was prioritized over disk space.


In [1]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import lightgbm as lgb
import joblib
from joblib import Parallel, delayed
import multiprocessing

import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
import gc

In [2]:
def fast_load_sparse(prefix):
    data = np.load(f"{prefix}/data.npy")
    indices = np.load(f"{prefix}/indices.npy")
    indptr = np.load(f"{prefix}/indptr.npy")
    shape = np.load(f"{prefix}/shape.npy")
    return sp.csr_matrix((data, indices, indptr), shape=shape)


In [3]:
import numpy as np
import scipy.sparse as sp
import lightgbm as lgb
import gc

# โหลด X_train, X_test (ใช้ฟังก์ชันเดิมของคุณ)
X_train = fast_load_sparse("X_train_raw")
X_test = fast_load_sparse("X_test_raw")

# อย่าลืมโหลด y
y_train = np.load("y_train.npy")
y_test = np.load("y_test.npy")

print(f"Data Loaded: Train {X_train.shape}, Test {X_test.shape}")

Data Loaded: Train (1261766, 9002), Test (140197, 9002)


In [7]:
train_data = lgb.Dataset(X_train, label=y_train)
val_data   = lgb.Dataset(X_test,  label=y_test, reference=train_data)

params = {
    "objective": "binary",
    "metric": "auc",
    "device_type": "cpu",   # force CPU
    "learning_rate": 0.01,
    "num_leaves": 63,
    "max_bin": 255,         # CPU usually can use larger bin
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "verbosity": -1,
    "num_threads": 5       # use all CPU cores
}

In [ ]:
import lightgbm as lgb
import numpy as np

# สร้างข้อมูลปลอมตัวอย่าง
X = np.random.rand(100, 10)
y = np.random.randint(0, 2, 100)
train_data = lgb.Dataset(X, label=y)

# ลองรันด้วย GPU
params = {
    "objective": "binary",
    "device": "gpu",  # หรือ "cuda"
    "verbose": -1
}

try:
    bst = lgb.train(params, train_data, num_boost_round=10)
    print("✅ LightGBM GPU is working perfectly!")
except Exception as e:
    print(f"❌ GPU failed: {e}")

In [8]:

num_rounds = 100


In [9]:
def tqdm_callback(pbar):
    def _callback(env):
        pbar.update(1)
        if env.evaluation_result_list:
            last_res = env.evaluation_result_list[-1]
            pbar.set_postfix({f"{last_res[0]}_{last_res[1]}": f"{last_res[2]:.4f}"})
    _callback.order = 10
    return _callback

print("Starting LightGBM CPU training...")

with tqdm(total=num_rounds, desc="Training LGBM (CPU)", unit="round") as pbar:
    model = lgb.train(
        params,
        train_data,
        num_boost_round=num_rounds,
        valid_sets=[val_data],
        valid_names=["valid"],
        callbacks=[
            lgb.early_stopping(30),
            lgb.log_evaluation(0),
            tqdm_callback(pbar)
        ],
    )

print("✅ CPU training complete.")

# Evaluate
y_pred = model.predict(X_test)
auc = roc_auc_score(y_test, y_pred)
print(f"Test AUC: {auc:.4f}")
print(classification_report(y_test, (y_pred > 0.5).astype(int),
                            target_names=["not 5-star", "5-star"]))

Starting LightGBM CPU training...


Training LGBM (CPU):   1%|          | 1/100 [03:08<5:11:15, 188.64s/round, valid_auc=0.7933]

Training until validation scores don't improve for 30 rounds


Training LGBM (CPU): 100%|██████████| 100/100 [10:56<00:00,  4.22s/round, valid_auc=0.8200] 

Did not meet early stopping. Best iteration is:
[100]	valid's auc: 0.820016


Training LGBM (CPU): 100%|██████████| 100/100 [10:56<00:00,  6.57s/round, valid_auc=0.8200]


✅ CPU training complete.
Test AUC: 0.8200
              precision    recall  f1-score   support

  not 5-star       0.92      0.30      0.45     38990
      5-star       0.79      0.99      0.88    101207

    accuracy                           0.80    140197
   macro avg       0.85      0.64      0.66    140197
weighted avg       0.82      0.80      0.76    140197



In [10]:
y_pred = model.predict(X_test)
auc    = roc_auc_score(y_test, y_pred)
print(f"\n  Test AUC: {auc:.4f}")
print(classification_report(y_test, (y_pred > 0.5).astype(int),
                             target_names=["not 5-star", "5-star"]))


  Test AUC: 0.8200
              precision    recall  f1-score   support

  not 5-star       0.92      0.30      0.45     38990
      5-star       0.79      0.99      0.88    101207

    accuracy                           0.80    140197
   macro avg       0.85      0.64      0.66    140197
weighted avg       0.82      0.80      0.76    140197



In [ ]:
joblib.dump(model,      "model.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")
print("\nSaved: model.pkl  vectorizer.pkl  recipe_vectors.pkl")